<a href="https://colab.research.google.com/github/mshinno26/UnderstandingAI/blob/main/neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the code from https://sirupsen.com/napkin/neural-net, use this to evolve the model with the suggested exercises in the article

Imports

In [15]:
from google.colab import drive
import torch
import torch.nn.functional as F
import librosa
import os
import numpy as np
from sklearn.model_selection import train_test_split

Google Drive mounting

In [16]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Universal constants

In [17]:
data_dir = "/content/drive/My Drive/Grade 12/AI/Numbers/Dataset"

n_mfcc = 13 # number of MFCC coefficients
max_len = 100 # number of time steps (change to match ideal sound file length)
num_classes = 10 # 0-9
input_size = n_mfcc * max_len # size of tensors

Class for MFCC conversion

In [18]:
class Converter:
  def __init__(self, n_mfcc):
    self._n_mfcc = n_mfcc

  def set_path(self, path):
    # Load audio
    self._audio, self._sr = librosa.load(path, sr=None)

  def convert(self, max_len):
    self._mfcc = librosa.feature.mfcc(y=self._audio, sr=self._sr, n_mfcc=self._n_mfcc)
    self.pad(max_len)
    return self.get_flat()

  def pad(self, max_len):
    # pad or truncate vectors to make them all the same length, regardless of audio file length
    length = self._mfcc.shape[1]
    if length < max_len:
        pad_size = max_len - length
        self._mfcc = np.pad(self._mfcc, ((0, 0), (0, pad_size)))
    else:
        self._mfcc = self._mfcc[:, :max_len]

  def get_flat(self):
    # Flatten from 2D to 1D vector
    return self._mfcc.flatten()

Convert sound files to MFCC vectors & store them

In [19]:
X = []
y = []

converter = Converter(n_mfcc)

# loop through audio files in each number's folder in the directory containing training set
for label in range(num_classes):
    folder = os.path.join(data_dir, str(label))

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        converter.set_path(path)
        mfcc = converter.convert(max_len)

        X.append(mfcc)
        y.append(label)

/tmp/ipykernel_2951/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2951/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_2951/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/l

Split training & testing sets, convert to tensors

In [20]:
# Convert to numpy array so scikit can work with them
X = np.array(X)
y = np.array(y)

# Normalize MFCC vectors (values can differ a lot, which wouldn't be good I guess?); looks very complicated because I need to normalize one sample at a time
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

# Split training & testing; "stratify=y" ensures that there is an equal number of each class in the sets; for example, 20% of zeroes and 20% of ones, as opposed to a random 20% of all numbers
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert from arrays to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# Normalize datatypes
X_train = X_train.float()
X_test = X_test.float()
y_train = y_train.long()
y_test = y_test.long()

print("X training shape:", X_train.shape)
print("X testing shape:", X_test.shape)
print("y training shape:", y_train.shape)
print("y testing shape:", y_test.shape)

X training shape: torch.Size([128, 1300])
X testing shape: torch.Size([32, 1300])
y training shape: torch.Size([128])
y testing shape: torch.Size([32])


Neural net class

In [23]:
class FCNN:
  def __init__(self, input_size, num_classes, hidden_size = 64, learning_rate = 0.01):
    self._input_size = input_size
    self._num_classes = num_classes
    self._hidden_size = hidden_size
    self._learning_rate = learning_rate

    # Initialize Weights for 2 Layers
    # Layer 1: 650 length flattened vector -> 64 hidden neurons
    self._w1 = torch.randn(self._input_size, self._hidden_size) * 0.01
    self._w1.requires_grad_()
    self._b1 = torch.zeros(self._hidden_size, requires_grad=True)

    # Layer 2: 64 hidden neurons -> 10 probabilities
    self._w2 = torch.randn(self._hidden_size, self._num_classes) * 0.01
    self._w2.requires_grad_()
    self._b2 = torch.zeros(self._num_classes, requires_grad=True)

  def classify(self, x):
    # Layer 1 with ReLU activation (the non-linear part)
    hidden = torch.relu(torch.matmul(x, self._w1) + self._b1)
    # Layer 2 (output)
    output = torch.matmul(hidden, self._w2) + self._b2
    return output

  def train(self, X_train, y_train):
    # Training Loop
    for epoch in range(5000):
      preds = self.classify(X_train)
      loss = F.cross_entropy(preds, y_train)

      loss.backward()

      with torch.no_grad():
        for param in [self._w1, self._b1, self._w2, self._b2]:
          param -= self._learning_rate * param.grad
          param.grad.zero_()

      if epoch % 50 == 0:
        print("Loss: ", loss.item())

  def test(self, X_test, y_test):
    with torch.no_grad():
      out = self.classify(X_test)
      print(out)

      predictions = torch.argmax(out, dim = 1)

      accuracy = (predictions == y_test).float().mean()
      print(predictions)
      print(y_test)

    return accuracy.item()

Train and test with existing dataset

In [24]:
fcnn = FCNN(input_size, num_classes)
fcnn.train(X_train, y_train)
accuracy = fcnn.test(X_test, y_test)
print("Accuracy: ", accuracy)

Loss:  2.302263021469116
Loss:  2.298908233642578
Loss:  2.295827865600586
Loss:  2.292229175567627
Loss:  2.287585973739624
Loss:  2.281463861465454
Loss:  2.2731637954711914
Loss:  2.26202130317688
Loss:  2.247051954269409
Loss:  2.226956605911255
Loss:  2.2003393173217773
Loss:  2.1659905910491943
Loss:  2.122994899749756
Loss:  2.071261167526245
Loss:  2.0117270946502686
Loss:  1.9461913108825684
Loss:  1.8768672943115234
Loss:  1.8058348894119263
Loss:  1.7346757650375366
Loss:  1.6644387245178223
Loss:  1.595803141593933
Loss:  1.52922785282135
Loss:  1.4649165868759155
Loss:  1.4029532670974731
Loss:  1.3434463739395142
Loss:  1.2863879203796387
Loss:  1.2317472696304321
Loss:  1.1793712377548218
Loss:  1.129136085510254
Loss:  1.080942153930664
Loss:  1.0346941947937012
Loss:  0.9902491569519043
Loss:  0.9475119113922119
Loss:  0.9064704775810242
Loss:  0.8669301867485046
Loss:  0.8289288282394409
Loss:  0.7923219203948975
Loss:  0.7571074366569519
Loss:  0.72319096326828
Loss:

Predict a singular sound

In [ ]:
single_folder_path =
path = ""

for file in os.listdir(single_folder_path):
  path = os.path.join(single_folder_path, file)

converter.set_path(path)
mfcc = converter.convert(max_len)

X = (mfcc - mfcc.mean()) / mfcc.std()
X = np.array(X)
X = torch.tensor(X, dtype=torch.float32)

pred = torch.argmax(fcnn.classify(X))
print("Predicted label: ", pred.item())